# 🏺 POI Data Inspector: "Thawing" the Pickle
**Goal:** This notebook is a diagnostic tool to "unpickle" the binary `manhattan_poi.pkl` file and convert it into a readable format. 

### 🎯 Objectives:
1. **Identify Column Names:** Resolve `KeyError` issues by finding the exact names for coordinate columns (e.g., `x`/`y` vs. `geometry` vs. `lat`/`lon`).
2. **Verify Unique IDs:** Determine which column (e.g., `osmid`, `node_id`, or `u`) maps back to our Manhattan Graph.
3. **Audit Task 2.5 Mapping:** Confirm that the OpenStreetMap tags we defined in `config.py` (like `amenity`, `shop`, and `leisure`) actually exist as columns in this dataset.

---
*Note: This is a one-time inspection to ensure the **OracleEngine** and **SymbolicSolver** are using the correct data keys.*

In [3]:
import os
import sys
import pickle
import pandas as pd
import pandas.core.indexes.base 

# --- THE UNIVERSAL PANDAS 2.x PATCH ---
try:
    import pandas.core.indexes.numeric
except ModuleNotFoundError:
    # If 'numeric' doesn't exist, we point the system to 'base'
    import pandas.core.indexes.base
    sys.modules['pandas.core.indexes.numeric'] = pandas.core.indexes.base

# Inject the missing Index types directly
pandas.core.indexes.base.Int64Index = pd.Index
pandas.core.indexes.base.UInt64Index = pd.Index
# ---------------------------------------

cities = ["manhattan", "pittsburgh", "philadelphia"]

for city in cities:
    print(f"\n🔍 --- Auditing City: {city.upper()} ---")
    poi_path = os.path.join('..', 'data', city, f'{city}_poi.pkl')
    
    if os.path.exists(poi_path):
        try:
            with open(poi_path, 'rb') as f:
                temp_df = pickle.load(f)
                
            print(f"✅ Loaded {len(temp_df)} POIs.")
            
            # KEY CHECK 1: The Prefix Test
            # Look at the first 3 OSMIDs to catch any patterns
            sample_ids = temp_df['osmid'].head(3).tolist()
            print(f"📊 Sample OSMID format: {sample_ids}")
            
            # KEY CHECK 2: Geometry format
            # Important: Different cities might use 'centroid' or 'geometry'
            geo_col = 'centroid' if 'centroid' in temp_df.columns else 'geometry'
            print(f"📍 Using Geometry Column: '{geo_col}'")
            
            # KEY CHECK 3: Coordinate existence
            if 'x' in temp_df.columns and 'y' in temp_df.columns:
                print("🌐 Found pre-calculated 'x' and 'y' columns.")
            else:
                print("⚠️ Missing 'x/y' columns - will need to derive from geometry.")

        except Exception as e:
            print(f"❌ Error loading {city}: {e}")
    else:
        print(f"❌ File not found for {city} at {poi_path}")


🔍 --- Auditing City: MANHATTAN ---


C:\Users\adan\AppData\Local\Temp\ipykernel_64844\2183206811.py:29: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  temp_df = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_64844\2183206811.py:29: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  temp_df = pickle.load(f)


✅ Loaded 20979 POIs.
📊 Sample OSMID format: ['#666', '#61785451', '#158801311']
📍 Using Geometry Column: 'centroid'
⚠️ Missing 'x/y' columns - will need to derive from geometry.

🔍 --- Auditing City: PITTSBURGH ---


C:\Users\adan\AppData\Local\Temp\ipykernel_64844\2183206811.py:29: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  temp_df = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_64844\2183206811.py:29: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  temp_df = pickle.load(f)


✅ Loaded 4998 POIs.
📊 Sample OSMID format: ['#34184938', '104919888', '#105987017']
📍 Using Geometry Column: 'centroid'
⚠️ Missing 'x/y' columns - will need to derive from geometry.

🔍 --- Auditing City: PHILADELPHIA ---
✅ Loaded 10302 POIs.
📊 Sample OSMID format: ['#109920298', '#157544809', '#157547044']
📍 Using Geometry Column: 'centroid'
⚠️ Missing 'x/y' columns - will need to derive from geometry.


C:\Users\adan\AppData\Local\Temp\ipykernel_64844\2183206811.py:29: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  temp_df = pickle.load(f)


In [4]:
# Check a few random node IDs in the Pittsburgh Graph
pitt_graph_path = "../data/pittsburgh/pittsburgh_graph.gpickle"
with open(pitt_graph_path, 'rb') as f:
    G_pitt = pickle.load(f)

print(f"Sample Pitt Nodes: {list(G_pitt.nodes())[:5]}")

C:\Users\adan\AppData\Local\Temp\ipykernel_64844\728948865.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  G_pitt = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_64844\728948865.py:4: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G_pitt = pickle.load(f)


Sample Pitt Nodes: ['1#34184938', '#34184938', '1#105987017', '#105987017', '1#153846392']


## 🔍 POI Data Inspection Results (Manhattan)
**Date:** 2026-03-11  
**File:** `manhattan_poi.pkl`

### 🛠️ Compatibility Requirements
The `.pkl` file was saved using a legacy version of Pandas (1.x). To load this in a **Pandas 2.0+** environment, the following "Monkey Patch" must be applied before `pickle.load()`:
* **Redirect Path:** Map `pandas.core.indexes.numeric` to `pandas.core.indexes.base`.
* **Restore Class:** Map `pandas.core.indexes.base.Int64Index` to `pd.Index`.

### 📊 Data Structure Summary
* **Total Columns:** 1,033
* **Primary ID:** `osmid` (Format: `#12345678`)
* **Coordinate System:** No standalone `x`/`y` columns. Data is stored in a **`geometry`** column as Shapely `POINT` objects.
    * *Extraction Method:* Use `.geometry.x` for Longitude and `.geometry.y` for Latitude.

### 🏷️ Key Mapping for Task 2.5/3.1
To resolve landmarks by keywords, we must filter using these specific column names found in the dataset:
| Feature Type | POI Column | Example Values |
| :--- | :--- | :--- |
| **Amenities** | `amenity` | `place_of_worship`, `restaurant`, `school` |
| **Shopping** | `shop` | `bakery`, `clothes`, `supermarket` |
| **Tourism** | `tourism` | `museum`, `attraction`, `viewpoint` |
| **Historic** | `historic` | `memorial`, `monument` |

---
**Note:** When calculating distances, ensure the Agent's coordinates (from the `.gpickle` graph) are in the same CRS (Coordinate Reference System) as these Points.

In [9]:
# Let's see which "Specificity" columns actually have data
important_cols = [
    'amenity', 'shop', 'office', 'religion', 
    'cuisine', 'brand', 'historic', 'tourism'
]

# This shows how many non-empty rows exist for each
print("--- Data Density Check ---")
for col in important_cols:
    if col in df.columns:
        non_nan = df[col].count()
        print(f"{col}: {non_nan} entries")

# Let's see a "Slice" of an Office or Place of Worship
print("\n--- Example: Diversity of 'place_of_worship' ---")
if 'religion' in df.columns:
    display(df[df['amenity'] == 'place_of_worship'][['name', 'religion', 'denomination']].head(5))

print("\n--- Example: Diversity of 'office' ---")
if 'office' in df.columns:
    display(df[df['office'].notna()][['name', 'office']].head(5))

--- Data Density Check ---
amenity: 11671 entries
shop: 3539 entries
office: 540 entries
religion: 234 entries
cuisine: 2673 entries
brand: 2333 entries
historic: 241 entries
tourism: 867 entries

--- Example: Diversity of 'place_of_worship' ---


,name,religion,denomination
79,Congregation Shaare Zedek,jewish,NaN
86,South Baptist Church,christian,baptist
87,First Ukrainian Assembly of God Church,christian,NaN
88,People's Home Church,christian,NaN
89,Lutheran Church In America,christian,lutheran



--- Example: Diversity of 'office' ---


,name,office
55,Council on Foreign Relations,association
107,The New York Foundling,ngo
222,Center for Jewish History,association
230,Association of the Bar of the City of New York,association
233,Permanent Mission of Egypt to the United Nations,diplomatic


In [11]:
import numpy as np

# 1. SETUP: Find a Synagogue and a Church from your actual data
synagogue = df[df['name'].str.contains("Shaare Zedek", na=False, case=False)].iloc[0]
church = df[df['name'].str.contains("South Baptist Church", na=False, case=False)].iloc[0]

# 2. MOCK AGENT POSITION: 
# We place the agent 0.0001 degrees away from the Synagogue 
# and 0.0005 degrees away from the Church.
# Mathematically, the Synagogue is "closer."
mock_agent_x = synagogue.geometry.x + 0.0001
mock_agent_y = synagogue.geometry.y + 0.0001

print(f"📍 Mock Agent is closest to: {synagogue['name']}")
print(f"⛪ Target Landmark: 'Church'")

# 3. RUN THE SCORING LOGIC (Internal logic of resolve_by_tags)
# We filter for 'place_of_worship'
test_df = df[df['amenity'] == 'place_of_worship'].copy()

# Calculate Distance Score using .centroid to handle both Points and Polygons
dist = ((test_df.geometry.centroid.x - mock_agent_x)**2 + (test_df.geometry.centroid.y - mock_agent_y)**2)**0.5
test_df['spatial_score'] = 1 / (1 + dist * 1000)

# Calculate Semantic Score for "Church"
test_df['semantic_score'] = test_df['name'].str.lower().str.contains("church", na=False).astype(int) * 2.0

test_df['final_score'] = test_df['spatial_score'] + test_df['semantic_score']

# 4. RESULTS
top_results = test_df.sort_values('final_score', ascending=False).head(5)
print("\n--- TOP 5 RANKED RESULTS ---")
display(top_results[['name', 'spatial_score', 'semantic_score', 'final_score']])

winner = top_results.iloc[0]['name']
if "Church" in winner:
    print(f"\n✅ SUCCESS: The Oracle chose '{winner}' because of its semantic match, even though it was further away!")
else:
    print(f"\n❌ FAILURE: The Oracle chose '{winner}' based on distance alone.")

📍 Mock Agent is closest to: Congregation Shaare Zedek
⛪ Target Landmark: 'Church'

--- TOP 5 RANKED RESULTS ---


,name,spatial_score,semantic_score,final_score
20142,Most Precious Blood Church,0.172632,2.0,2.172632
20157,Church of the Transfiguration,0.145857,2.0,2.145857
9287,The First Chinese Baptist Church,0.134278,2.0,2.134278
22058,St. James Church,0.111934,2.0,2.111934
21835,Saint Peter's Church,0.106939,2.0,2.106939



✅ SUCCESS: The Oracle chose 'Most Precious Blood Church' because of its semantic match, even though it was further away!


In [13]:
# Define what we are looking for
search_term = "Joe's"
tag_term = "dim_sum"

# 1. Search by Name
name_matches = df[df['name'].str.contains(search_term, case=False, na=False)]

# 2. Search by Tag (across all columns)
tag_matches = df[df.apply(lambda row: any(tag_term in str(val).lower() for val in row.values), axis=1)]

print(f"--- Results for '{search_term}' ---")
print(f"Found {len(name_matches)} matches in 'name' column.")
if not name_matches.empty:
    display(name_matches[['name', 'amenity', 'cuisine']].head(5))

print(f"\n--- Results for tag '{tag_term}' ---")
print(f"Found {len(tag_matches)} matches across all tags.")
if not tag_matches.empty:
    display(tag_matches[['name', 'amenity', 'cuisine']].head(5))

--- Results for 'Joe's' ---
Found 19 matches in 'name' column.


,name,amenity,cuisine
2456,Joe's Pizza,restaurant,pizza
2478,Trader Joe's,NaN,NaN
3286,Trader Joe's,NaN,NaN
3287,Trader Joe's Wine Shop,NaN,NaN
3306,Joe's Shanghai,restaurant,chinese



--- Results for tag 'dim_sum' ---
Found 2 matches across all tags.


,name,amenity,cuisine
3695,Tim Ho Wan,restaurant,dim_sum; chinese
8179,88 palace,restaurant,dim_sum


In [14]:
# 1. SETUP: A generic Chinese restaurant vs a specific Dim Sum spot
# Assuming Joe's is in our data
dim_sum_spot = df[df['cuisine'] == 'dim_sum'].iloc[0]
generic_chinese = df[(df['cuisine'] == 'chinese') & (df['cuisine'] != 'dim_sum')].iloc[0]

# 2. MOCK AGENT: Closer to the generic Chinese spot
mock_agent_x = generic_chinese.geometry.centroid.x + 0.0001
mock_agent_y = generic_chinese.geometry.centroid.y + 0.0001

print(f"📍 Agent is closer to: {generic_chinese['name']} (Generic Chinese)")
print(f"🥡 Target: 'Dim Sum'")

# 3. SCORING
test_df = df[df['amenity'] == 'restaurant'].copy()
dist = ((test_df.geometry.centroid.x - mock_agent_x)**2 + (test_df.geometry.centroid.y - mock_agent_y)**2)**0.5

# Spatial
test_df['spatial_score'] = 1 / (1 + dist * 1000)

# Deep Semantic (Checks ALL columns for the string "dim_sum")
test_df['semantic_score'] = test_df.apply(
    lambda row: 1.5 if any("dim_sum" in str(val).lower() for val in row.values) else 0.0, axis=1
)

test_df['final_score'] = test_df['spatial_score'] + test_df['semantic_score']
winner = test_df.sort_values('final_score', ascending=False).iloc[0]

print(f"\n🏆 Result: The Oracle chose '{winner['name']}'")
print(f"📊 Scores -> Spatial: {winner['spatial_score']:.2f}, Semantic: {winner['semantic_score']:.2f}")

📍 Agent is closer to: Wa Lung Kitchen (Generic Chinese)
🥡 Target: 'Dim Sum'

🏆 Result: The Oracle chose '88 palace'
📊 Scores -> Spatial: 0.07, Semantic: 1.50


### Drop 'amenity' and see what else is densely populated

In [16]:
non_amenity_df = df.drop(columns=['amenity'], errors='ignore')
print(non_amenity_df.notna().sum().sort_values(ascending=False).head(40))

cellids                20979
unique_id              20979
osmid                  20979
element_type           20979
centroid               20979
geometry               20979
name                   14076
addr:street             9095
addr:housenumber        8999
addr:postcode           7093
website                 5242
nodes                   4851
addr:city               4508
phone                   4325
opening_hours           4072
shop                    3539
addr:state              3237
capacity                2982
building                2950
cuisine                 2673
nycdoitt:bin            2541
brand                   2333
height                  2231
brand:wikidata          2216
brand:wikipedia         2180
cityracks.rackid        1916
cityracks.large         1915
cityracks.small         1915
cityracks.street        1915
cityracks.housenum      1915
operator                1831
wikidata                1796
leisure                 1411
wheelchair              1406
start_date    

Multi-City logic

In [1]:
# List of cities to audit
cities = ["manhattan", "pittsburgh", "philadelphia"]

for city in cities:
    print(f"\n🔍 --- Auditing City: {city.upper()} ---")
    poi_path = os.path.join('..', 'data', city, f'{city}_poi.pkl')
    
    if os.path.exists(poi_path):
        with open(poi_path, 'rb') as f:
            # Use the same patch as before
            temp_df = pickle.load(f)
            
        print(f"✅ Loaded {len(temp_df)} POIs.")
        
        # KEY CHECK 1: The Prefix Test
        # We need to see if 'osmid' starts with '#', '1#', or is just an integer
        sample_id = str(temp_df['osmid'].iloc[0])
        print(f"📊 Sample OSMID format: '{sample_id}'")
        
        # KEY CHECK 2: Column Audit
        # Ensure the columns we rely on in config.py exist here
        essential_cols = ['amenity', 'shop', 'leisure', 'name']
        found_cols = [c for c in essential_cols if c in temp_df.columns]
        print(f"🏷️ Essential Tags Found: {found_cols}")
        
        # KEY CHECK 3: Geometry format
        # Verify if it uses 'centroid' (like Manhattan) or something else
        if 'centroid' in temp_df.columns:
            print("📍 Geometry Column: 'centroid'")
        elif 'geometry' in temp_df.columns:
            print("📍 Geometry Column: 'geometry'")
            
    else:
        print(f"❌ File not found for {city} at {poi_path}")


🔍 --- Auditing City: MANHATTAN ---


NameError: name 'os' is not defined